<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 5: *Subset and Split*
##### Version Number: 4.0
---
### Contents  
> 1. *Filter Dates for modeling*
> 2. *Split and Scale Data*
> 3. *Export Data*
---
### Notes
---
### Inputs
- `X`,`y`,`details` - Modeling sets
---
### Outputs 
- `X_scaled`,`y_reduced`,`details_reduced` - Scaled Modeling sets (sometimes with reduced modeling sets to preserve processor)
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

###  Load Data

In [3]:
samples = pd.read_csv("../data/processed/variable_selection_output.csv")

In [4]:
samples['Target_Ignition'].value_counts()

Target_Ignition
0    565490
1     29507
2     13883
Name: count, dtype: int64

In [5]:
samples['Target_Spread'].value_counts()

Target_Spread
0    587078
1     12189
2      9613
Name: count, dtype: int64

In [6]:
samples['Target_Damage'].value_counts()

Target_Damage
0    608352
1       528
Name: count, dtype: int64

## Filter Dates for modeling

- **01/01/2018 - 12/31/2024** Dates to train the models
- **01/01/2025 - 01/23/2025** Dates for evaluating model performance on unseen data

In [7]:
samples['Date'] = pd.to_datetime(samples['Date'])

# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

# Boolean mask for dates within the range
mask = (samples['Date'].dt.date >= FIRST_DATE) & (samples['Date'].dt.date <= LAST_DATE)

pal = samples.loc[~mask]

# Keep only rows within the range
model_samples = samples.loc[mask].copy()

### (OPTIONAL) Subset classes to save processor
- All classes are downsized for working on models

In [8]:
model_samples['Date'] = pd.to_datetime(model_samples['Date'])

ignition_reduced = subset_df(model_samples,'Target_Ignition',50000)
spread_reduced = subset_df(model_samples,'Target_Spread',50000)
damage_reduced = subset_df(model_samples,'Target_Damage',50000)

In [9]:
ignition_reduced['Target_Ignition'].value_counts()

Target_Ignition
0    50000
1    29295
2    13744
Name: count, dtype: int64

In [10]:
spread_reduced['Target_Spread'].value_counts()

Target_Spread
0    50000
1    12060
2     9533
Name: count, dtype: int64

In [11]:
damage_reduced['Target_Damage'].value_counts()

Target_Damage
0    50000
1      525
Name: count, dtype: int64

In [12]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date', 'grid_id',
       'geometry', 'fire_count', 'total_fire_damage','acres','area_in_cali','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','centroid_northing','centroid_easting','Target_Damage','Target_Ignition',
               'Target_Spread','Year','fires_missing_acres','fires_missing_damage']

y_ignition = ignition_reduced['Target_Ignition']
y_spread = spread_reduced['Target_Spread']
y_damage = damage_reduced['Target_Damage']

pal_y = pal[['Target_Ignition','Target_Spread','Target_Damage']]

X_ignition = ignition_reduced.drop(columns=text_columns)
X_spread = spread_reduced.drop(columns=text_columns)
X_damage = damage_reduced.drop(columns=text_columns)

pal_X = pal.drop(columns=text_columns)

details_ignition = ignition_reduced[text_columns]
details_spread = spread_reduced[text_columns]
details_damage = damage_reduced[text_columns]

pal_details = pal[text_columns]

### Rank variables by correlation strength

In [13]:
rank_variables_by_correlation(X_ignition,y_ignition)

,Feature,Correlation
0,road_length_meters,0.479026
1,power_line_meters,0.468201
2,road_density,0.448226
3,power_line_density,0.443768
4,median_income,0.439303
...,...,...
108,Season_Fall,0.013874
109,Standardized Precipitation Evapotranspiration ...,0.012523
110,Standardized Precipitation Index 30-Day,0.011953
111,Standardized Precipitation Index 180-Day,0.003048


In [14]:
rank_variables_by_correlation(X_spread,y_spread)

,Feature,Correlation
0,influence_zone,0.323982
1,Solar Radiation 7 Day Avg,0.290256
2,Solar Radiation,0.270451
3,intermix_zone,0.265644
4,Season_Summer,0.259857
...,...,...
108,dominant_section_description_Northern Californ...,-0.005551
109,dominant_section_description_Southern Cascades,-0.004519
110,Standardized Precipitation Index 30-Day,-0.002899
111,Season_Fall,-0.001479


In [15]:
rank_variables_by_correlation(X_damage,y_damage)

,Feature,Correlation
0,Season_Summer,0.109021
1,Actual Evapotranspiration,0.083651
2,Solar Radiation 7 Day Avg,0.082895
3,Daily Minimum Air Temperature,0.082708
4,Daily Maximum Air Temperature,0.081099
...,...,...
108,Standardized Precipitation Index 180-Day,-0.001383
109,total_housing,0.001221
110,population_density,-0.001168
111,Standardized Precipitation Index 30-Day,-0.000671


---

## 3. Export Data

In [16]:
X_ignition.to_csv('../data/processed/X_ignition.csv', index=False)
X_spread.to_csv('../data/processed/X_spread.csv', index=False)
X_damage.to_csv('../data/processed/X_damage.csv', index=False)

y_ignition.to_csv('../data/processed/y_ignition.csv', index=False)
y_spread.to_csv('../data/processed/y_spread.csv', index=False)
y_damage.to_csv('../data/processed/y_damage.csv', index=False)

details_ignition.to_csv('../data/processed/details_ignition.csv', index=False)
details_spread.to_csv('../data/processed/details_spread.csv', index=False)
details_damage.to_csv('../data/processed/details_damage.csv', index=False)

pal_X.to_csv('../data/processed/pal_X.csv', index=False)
pal_y.to_csv('../data/processed/pal_y.csv', index=False)
pal_details.to_csv('../data/processed/pal_details.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/


In [17]:
X_ignition.columns

Index(['Burning Index', 'Energy Release Component',
       'Actual Evapotranspiration', '100-hour Dead Fuel Moisture',
       '1000-hour Dead Fuel Moisture', 'Precipitation',
       'Maximum Relative Humidity', 'Minimum Relative Humidity',
       'Specific Humidity', 'Solar Radiation',
       ...
       'dominant_section_description_Sierra Nevada Foothills',
       'dominant_section_description_Sonoran Desert',
       'dominant_section_description_Southeastern Great Basin',
       'dominant_section_description_Southern California Coast',
       'dominant_section_description_Southern California Mountains and Valleys',
       'dominant_section_description_Southern Cascades', 'Season_Fall',
       'Season_Spring', 'Season_Summer', 'Season_Winter'],
      dtype='object', length=113)